# Classical ML Lab — AI Lab Instructor Notebook

*Classical ML · Medium*

Build an end-to-end classical ML pipeline the way FAANG expects: correct splitting, leakage-safe preprocessing, cross-validation, baselines, and clean reporting.

**Outcome.** Students can:
- load Kaggle Titanic reliably,
- build a sklearn `Pipeline` + `ColumnTransformer`,
- compare multiple models under cross-validation,
- perform slice-based error analysis.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Classical ML Lab — Student (Titanic)

Build an end-to-end sklearn pipeline with correct validation and error analysis.

## Section 0 — Load Titanic (hosted CSV + fallback)
Primary: `data/titanic/train.csv`
Fallback: `data/titanic/train.csv`

If both are unavailable, a tiny synthetic dataset is used so the notebook runs.

In [ ]:
$10

**Why this works.** # Classical ML Lab — Instructor (Titanic)

## Task 2: Create train/holdout split
*Section: Holdout split (untouched)*

## Section 1 — Holdout split (untouched)

### Task 1.1: Create train/holdout split

**Hints**
- stratify on Survived.
- stratify on Survived

In [ ]:
y = df['Survived'].astype(int)
X = df.drop(columns=['Survived'])

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print('train/holdout', len(X_train), len(X_holdout))

In [ ]:
# Checks
check('holdout_nonempty', len(X_holdout) > 0)

## Task 3: Majority-class baseline
*Section: Baselines*

## Section 2 — Baselines

### Task 2.1: Majority-class baseline

In [ ]:
# Majority baseline
majority = int(y_train.mean() >= 0.5)
pred = np.full(len(y_holdout), majority)
print('majority_acc', accuracy_score(y_holdout, pred))

## Task 4: Define feature columns
*Section: Preprocessing pipeline*

## Section 3 — Preprocessing pipeline

### Task 3.1: Define feature columns
Use numeric + categorical columns.

In [ ]:
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

## Task 5: Compare models with StratifiedKFold
*Section: CV model comparison*

## Section 4 — CV model comparison

### Task 4.1: Compare models with StratifiedKFold

Models:
- LogisticRegression
- RandomForestClassifier
- GradientBoostingClassifier

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
models = {
    'lr': LogisticRegression(max_iter=5000),
    'rf': RandomForestClassifier(n_estimators=300, random_state=0),
    'gb': GradientBoostingClassifier(random_state=0),
}
scoring = {'acc': 'accuracy'}

cv_summary = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False)
    cv_summary.append((name, float(scores['test_acc'].mean()), float(scores['test_acc'].std())))

print('CV summary (name, mean, std)')
for row in cv_summary:
    print(row)

**Why this works.** # Rationale: Cross Validation is a safeguard against overfitting and a tool for better hyperparameter selection, ensuring that final performance is a more honest reflection of real-world generalization.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
models = {
    'lr': LogisticRegression(max_iter=5000),
    'rf': RandomForestClassifier(n_estimators=300, random_state=0),
    'gb': GradientBoostingClassifier(random_state=0),
}
scoring = {'acc': 'accuracy'}

cv_summary = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=False)
    cv_summary.append((name, float(scores['test_acc'].mean()), float(scores['test_acc'].std())))

print('CV summary (name, mean, std)')
for row in cv_summary:
    print(row)

## Task 6: Pick best model and evaluate on holdout
*Section: Final fit + holdout evaluation*

## Section 5 — Final fit + holdout evaluation

### Task 5.1: Pick best model and evaluate on holdout

In [ ]:
# Pick best by mean CV acc
best_name = sorted(cv_summary, key=lambda x: x[1], reverse=True)[0][0]
best_model = models[best_name]
print('best_model', best_name)

pipe = Pipeline(steps=[('preprocess', preprocess), ('model', best_model)])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_holdout)
print('holdout_acc', accuracy_score(y_holdout, pred))

## Task 7: Slice performance by Sex/Pclass
*Section: Slice-based error analysis*

## Section 6 — Slice-based error analysis

### Task 6.1: Slice performance by Sex/Pclass

In [ ]:
holdout_df = X_holdout.copy()
holdout_df['y_true'] = y_holdout.values
holdout_df['y_pred'] = pred
holdout_df['correct'] = (holdout_df['y_true'] == holdout_df['y_pred']).astype(int)

if 'Sex' in holdout_df.columns:
    print('acc_by_sex')
    print(holdout_df.groupby('Sex')['correct'].mean())
if 'Pclass' in holdout_df.columns:
    print('acc_by_pclass')
    print(holdout_df.groupby('Pclass')['correct'].mean())

# Rationale: slice analysis reveals failure modes hidden by overall accuracy.

In [ ]:
# Checks
check('slices_computed', 'correct' in holdout_df.columns)

**Why this works.** slice analysis reveals failure modes hidden by overall accuracy.